# 2026 March Madness Bracket Predictions 🏀

Use this notebook to fill out your 2026 NCAA Men's Tournament bracket based on the Cinderella System model predictions.

This shows:
1. **Team Power Rankings** — All 68 tournament teams ranked by model strength
2. **First Round Matchups** — Region-by-region predictions for the Round of 64
3. **Full Simulated Bracket** — Round-by-round predicted winners through the Championship
4. **Cinderella Watch** — Teams with the highest "heat" that could pull upsets
5. **Matchup Explorer** — Look up any head-to-head matchup

---

In [53]:
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np
import polars as pl

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
BASE = ROOT / "artifacts" / "latest"

# Load submission predictions
submission = pl.read_csv(BASE / "submission.csv")

# Load team metadata
teams = pl.read_csv(ROOT / "data" / "MTeams.csv").select("TeamID", "TeamName")
tid_to_name = dict(zip(teams["TeamID"].to_list(), teams["TeamName"].to_list()))

# Load 2026 seeds and slots
seeds_2026 = pl.read_csv(ROOT / "data" / "MNCAATourneySeeds.csv").filter(pl.col("Season") == 2026)
slots_2026 = pl.read_csv(ROOT / "data" / "MNCAATourneySlots.csv").filter(pl.col("Season") == 2026)

# Load team features for additional context
team_features = pl.read_parquet(BASE / "gold" / "team_season_features.parquet")
m_2026 = team_features.filter((pl.col("sex") == "M") & (pl.col("season") == 2026))

# Build seed → team mapping
seed_to_team = {}
seed_to_name = {}
for row in seeds_2026.iter_rows(named=True):
    tid = row["TeamID"]
    seed_to_team[row["Seed"]] = tid
    seed_to_name[row["Seed"]] = tid_to_name.get(tid, str(tid))

# Build probability lookup from submission
def get_pred(tid_a: int, tid_b: int) -> float:
    """Get P(lower-ID team wins) from submission."""
    low, high = min(tid_a, tid_b), max(tid_a, tid_b)
    match = submission.filter(pl.col("ID") == f"2026_{low}_{high}")
    if match.height > 0:
        return float(match["Pred"][0])
    return 0.5

# Region names
REGION_NAMES = {"W": "West", "X": "East", "Y": "South", "Z": "Midwest"}

print(f"Loaded {seeds_2026.height} seeded teams across 4 regions")
print(f"Loaded {slots_2026.height} bracket slots")
print(f"Submission has {submission.height:,} predictions")

Loaded 68 seeded teams across 4 regions
Loaded 67 bracket slots
Submission has 132,133 predictions


---

## Team Power Rankings

All 68 tournament teams ranked by season-end ELO rating. Higher ELO = stronger team based on regular season performance. Heat scores show recent momentum (positive = outperforming expectations).

In [54]:
# Power rankings: all 68 teams sorted by ELO
power = (
    m_2026
    .join(seeds_2026.select(
        pl.col("TeamID").cast(pl.Int64).alias("team_id"),
        pl.col("Seed"),
    ), on="team_id", how="inner")
    .join(teams.rename({"TeamID": "team_id"}), on="team_id", how="left")
    .sort("season_end_elo", descending=True)
)

print(f"{'#':>3} {'Seed':<6} {'Team':<22} {'ELO':>7} {'Record':>8} {'WR':>6} {'Margin':>7} {'H-1g':>7} {'H-3g':>7} {'H-5g':>7}")
print("─" * 90)
for i, row in enumerate(power.iter_rows(named=True), 1):
    record = f"{row['wins']}-{row['losses']}"
    h1 = row['pre_tourney_heat_1g'] or 0
    h3 = row['pre_tourney_heat_3g'] or 0
    h5 = row['pre_tourney_heat_5g'] or 0
    elo = row['season_end_elo'] or 1500
    print(f"{i:>3} {row['Seed']:<6} {row['TeamName']:<22} {elo:>7.0f} {record:>8} {row['win_rate']:>5.1%} {row['avg_margin']:>+7.1f} {h1:>+7.1f} {h3:>+7.1f} {h5:>+7.1f}")

  # Seed   Team                       ELO   Record     WR  Margin    H-1g    H-3g    H-5g
──────────────────────────────────────────────────────────────────────────────────────────
  1 W01    Duke                      1738     32-2 94.1%   +19.1    +4.7    +2.0   +10.3
  2 Z01    Arizona                   1732     32-2 94.1%   +17.4    -2.4    +3.1    +8.0
  3 Y01    Michigan                  1716     31-3 91.2%   +17.6    -3.4    -0.4    +0.6
  4 Z03    Gonzaga                   1691     30-3 90.9%   +19.1    -1.2    +4.1    +4.1
  5 Y03    Virginia                  1687     29-5 85.3%   +12.2   +18.5    +4.7    -3.2
  6 W05    St John's                 1685     28-6 82.4%   +11.6    +3.1    +2.1    +5.8
  7 Y11a   Miami OH                  1677     28-1 96.6%   +10.9   -11.0   -11.1    -7.2
  8 W02    Connecticut               1669     29-5 85.3%   +12.4    +4.9    -0.6    +5.1
  9 X02    Houston                   1666     28-6 82.4%   +14.3   +19.2    +7.1   +11.6
 10 X01    Florida

---

## Round of 64 — First Round Matchups by Region

Each region has 8 first-round games (1v16, 2v15, 3v14, ... 8v9). The model predicts the probability the higher-seeded team (stronger seed = lower number) wins.

**⚠️ Upset Alert** = model gives the lower-seeded team (underdog) a >35% chance of winning.

In [55]:
# First round matchups by region
first_round_slots = slots_2026.filter(pl.col("Slot").str.starts_with("R1")).sort("Slot")

for region_code, region_name in REGION_NAMES.items():
    region_slots = first_round_slots.filter(
        pl.col("Slot").str.contains(f"R1{region_code}")
    ).sort("Slot")

    print(f"\n{'=' * 78}")
    print(f"  {region_name.upper()} REGION ({region_code})")
    print(f"{'=' * 78}")
    print(f"{'Slot':<7} {'Strong':<22} {'vs':^4} {'Weak':<22} {'P(Strong)':>10} {'Pick'}")
    print("─" * 78)

    for row in region_slots.iter_rows(named=True):
        strong_seed = row["StrongSeed"]
        weak_seed = row["WeakSeed"]
        strong_tid = seed_to_team.get(strong_seed, 0)
        weak_tid = seed_to_team.get(weak_seed, 0)
        strong_name = f"({strong_seed[-2:]}) {seed_to_name.get(strong_seed, '?')}"
        weak_name = f"({weak_seed[-2:]}) {seed_to_name.get(weak_seed, '?')}"

        if strong_tid and weak_tid:
            # Get probability that the strong seed wins
            low_id = min(strong_tid, weak_tid)
            high_id = max(strong_tid, weak_tid)
            p_low_wins = get_pred(strong_tid, weak_tid)
            # If strong_tid is the low_id, p_strong = p_low_wins
            # If strong_tid is the high_id, p_strong = 1 - p_low_wins
            p_strong = p_low_wins if strong_tid == low_id else 1 - p_low_wins

            upset = "⚠️ UPSET ALERT" if p_strong < 0.65 else ""
            winner = strong_name if p_strong >= 0.5 else weak_name
            print(f"{row['Slot']:<7} {strong_name:<22} {'vs':^4} {weak_name:<22} {p_strong:>9.1%} {winner}  {upset}")
        else:
            print(f"{row['Slot']:<7} {strong_seed:<22} {'vs':^4} {weak_seed:<22} {'N/A':>10}")


  WEST REGION (W)
Slot    Strong                  vs  Weak                    P(Strong) Pick
──────────────────────────────────────────────────────────────────────────────
R1W1    (01) Duke               vs  (16) Siena                 97.5% (01) Duke  
R1W2    (02) Connecticut        vs  (15) Furman                97.5% (02) Connecticut  
R1W3    (03) Michigan St        vs  (14) N Dakota St           81.4% (03) Michigan St  
R1W4    (04) Kansas             vs  (13) Cal Baptist           86.6% (04) Kansas  
R1W5    (05) St John's          vs  (12) Northern Iowa         76.2% (05) St John's  
R1W6    (06) Louisville         vs  (11) South Florida         77.6% (06) Louisville  
R1W7    (07) UCLA               vs  (10) UCF                   44.7% (10) UCF  ⚠️ UPSET ALERT
R1W8    (08) Ohio St            vs  (09) TCU                   68.5% (08) Ohio St  

  EAST REGION (X)
Slot    Strong                  vs  Weak                    P(Strong) Pick
──────────────────────────────────────────

---

## Full Bracket Simulation — Predicted Winners Through Each Round

The model simulates 100,000 brackets using Monte Carlo simulation. For each game, it uses the pairwise win probability to randomly determine a winner, then advances them. The **most likely winner** of each slot across all simulations is shown below.

Round-by-round, region-by-region:

In [56]:
import re

# Run the full bracket simulation deterministically (pick highest-probability winner each round)
# and also show the simulation-based probabilities

from mm26.pipeline import _simulate_bracket, _build_prob_lookup

prob_lookup = _build_prob_lookup(submission)

def seed_num(seed_str: str) -> int:
    """Extract numeric seed from seed string (e.g. 'W01' -> 1, 'Y11a' -> 11)."""
    m = re.search(r"\d+", seed_str)
    return int(m.group()) if m else 0

# Run simulation
print("Running 100,000 bracket simulations...\n")
sim_preds = _simulate_bracket(seeds_2026, slots_2026, prob_lookup, n_sims=100_000)

# Also do a deterministic bracket walkthrough using pairwise probabilities
# Process slots round by round
slot_winner = {}  # slot_key -> (team_id, team_name, seed)

# Initialize seeds
for seed, tid in seed_to_team.items():
    slot_winner[seed] = (tid, tid_to_name.get(tid, str(tid)), seed)

# Sort slots: play-in games FIRST (no "R" prefix), then R1, R2, ... R6
def slot_sort_key(slot_dict):
    slot = slot_dict["Slot"]
    if slot.startswith("R"):
        return (int(slot[1]), slot)
    return (0, slot)  # Play-in games before round 1

sorted_slots = sorted(
    slots_2026.iter_rows(named=True),
    key=slot_sort_key
)

# Track results by round
round_labels = {
    "R1": "Round of 64",
    "R2": "Round of 32",
    "R3": "Sweet 16",
    "R4": "Elite 8",
    "R5": "Final Four",
    "R6": "Championship",
}

round_results = {label: [] for label in round_labels.values()}

for slot_info in sorted_slots:
    slot = slot_info["Slot"]
    strong_key = slot_info["StrongSeed"]
    weak_key = slot_info["WeakSeed"]

    strong = slot_winner.get(strong_key)
    weak = slot_winner.get(weak_key)

    if strong is None or weak is None:
        continue

    strong_tid, strong_name, strong_seed = strong
    weak_tid, weak_name, weak_seed = weak

    # Get prediction
    low_id = min(strong_tid, weak_tid)
    high_id = max(strong_tid, weak_tid)
    p_low = prob_lookup.get((low_id, high_id), 0.5)
    p_strong = p_low if strong_tid == low_id else 1 - p_low

    if p_strong >= 0.5:
        winner = (strong_tid, strong_name, strong_seed)
        loser_name, loser_seed = weak_name, weak_seed
        win_prob = p_strong
    else:
        winner = (weak_tid, weak_name, weak_seed)
        loser_name, loser_seed = strong_name, strong_seed
        win_prob = 1 - p_strong

    slot_winner[slot] = winner

    # Skip play-in games (slots not starting with "R") for display
    round_key = slot[:2]
    round_name = round_labels.get(round_key)
    if round_name is None:
        continue

    # Determine region
    region_char = slot[2] if len(slot) > 2 and slot[2] in "WXYZ" else ""

    round_results[round_name].append({
        "region": REGION_NAMES.get(region_char, ""),
        "winner_name": winner[1],
        "winner_seed": winner[2],
        "loser_name": loser_name,
        "loser_seed": loser_seed,
        "win_prob": win_prob,
        "slot": slot,
    })

# Verify game counts
expected = {"Round of 64": 32, "Round of 32": 16, "Sweet 16": 8, "Elite 8": 4, "Final Four": 2, "Championship": 1}
for rnd, exp in expected.items():
    actual = len(round_results[rnd])
    if actual != exp:
        print(f"⚠️  {rnd}: expected {exp} games, got {actual}")

# Display round by round
for round_name, results in round_results.items():
    if not results:
        continue
    print(f"\n{'═' * 78}")
    print(f"  {round_name.upper()} ({len(results)} games)")
    print(f"{'═' * 78}")

    # Group by region if applicable
    by_region: dict[str, list] = {}
    for r in results:
        rg = r["region"] or "National"
        by_region.setdefault(rg, []).append(r)

    for region, games in sorted(by_region.items()):
        if region != "National":
            print(f"\n  {region}:")
        for g in games:
            ws = seed_num(g["winner_seed"])
            ls = seed_num(g["loser_seed"])
            upset = " 🔥 UPSET" if ws > ls else ""
            print(f"    ({ws:>2}) {g['winner_name']:<20} over ({ls:>2}) {g['loser_name']:<20} [{g['win_prob']:.1%}]{upset}")

# Championship result
CHAMP_SLOT = "R6CH"
if CHAMP_SLOT in slot_winner:
    champ = slot_winner[CHAMP_SLOT]
    print(f"\n{'🏆' * 5}")
    print(f"  PREDICTED CHAMPION: ({seed_num(champ[2]):>2}) {champ[1]}")
    print(f"{'🏆' * 5}")

Running 100,000 bracket simulations...


══════════════════════════════════════════════════════════════════════════════
  ROUND OF 64 (32 games)
══════════════════════════════════════════════════════════════════════════════

  East:
    ( 1) Florida              over (16) Prairie View         [97.5%]
    ( 2) Houston              over (15) Idaho                [97.5%]
    ( 3) Illinois             over (14) Penn                 [92.2%]
    ( 4) Nebraska             over (13) Troy                 [79.2%]
    ( 5) Vanderbilt           over (12) McNeese St           [63.7%]
    ( 6) North Carolina       over (11) VCU                  [81.0%]
    ( 7) St Mary's CA         over (10) Texas A&M            [71.6%]
    ( 9) Iowa                 over ( 8) Clemson              [65.4%] 🔥 UPSET

  Midwest:
    ( 1) Arizona              over (16) LIU Brooklyn         [97.5%]
    ( 2) Purdue               over (15) Queens NC            [90.8%]
    ( 3) Gonzaga              over (14) Kennesaw         

---

## Cinderella Watch 🔮

These are the teams with the **highest pre-tournament heat scores** — teams that have been outperforming their ELO expectations heading into March. The Cinderella System thesis says these teams are most likely to pull upsets.

"Heat" measures per-game over-performance (actual margin minus ELO-expected margin), averaged over the last 5 games.

In [57]:
import re

# Cinderella Watch — hottest lower seeds entering the tournament
cinderella_candidates = (
    power
    .with_columns(
        pl.col("Seed").str.extract(r"(\d+)", 1).cast(pl.Int64).alias("seed_num_col"),
    )
    .filter(pl.col("seed_num_col") >= 7)  # Focus on teams seeded 7 or worse (upset candidates)
    .sort("pre_tourney_heat_5g", descending=True)
)

print("🔮 CINDERELLA CANDIDATES — Lower Seeds with Highest Heat\n")
print(f"{'Seed':<6} {'Team':<22} {'ELO':>7} {'Record':>8} {'H-5g':>7} {'H-3g':>7} {'H-1g':>7} {'Margin':>7}")
print("─" * 80)

for row in cinderella_candidates.head(15).iter_rows(named=True):
    h1 = row['pre_tourney_heat_1g'] or 0
    h3 = row['pre_tourney_heat_3g'] or 0
    h5 = row['pre_tourney_heat_5g'] or 0
    record = f"{row['wins']}-{row['losses']}"
    elo = row['season_end_elo'] or 1500
    print(f"{row['Seed']:<6} {row['TeamName']:<22} {elo:>7.0f} {record:>8} {h5:>+7.1f} {h3:>+7.1f} {h1:>+7.1f} {row['avg_margin']:>+7.1f}")

# Show their first-round matchup predictions
print(f"\n{'─' * 80}")
print(f"\nFirst-Round Upset Probabilities for Top Cinderella Candidates:\n")

for row in cinderella_candidates.head(10).iter_rows(named=True):
    seed = row["Seed"]
    region = seed[0]
    sn = int(re.search(r"\d+", seed).group())
    opp_sn = 17 - sn  # e.g., 12 seed plays 5 seed
    opp_seed = f"{region}{opp_sn:02d}"

    tid = row["team_id"]
    opp_tid = seed_to_team.get(opp_seed, 0)
    opp_name = seed_to_name.get(opp_seed, "?")

    if opp_tid:
        low_id = min(tid, opp_tid)
        high_id = max(tid, opp_tid)
        p_low = get_pred(tid, opp_tid)
        # P(this team wins): depends on whether this team is low or high
        p_win = p_low if tid == low_id else 1 - p_low
        icon = "🔥" if p_win > 0.35 else "  "
        print(f"  {icon} ({sn:>2}) {row['TeamName']:<20} vs ({opp_sn:>2}) {opp_name:<20}  P(upset) = {p_win:.1%}")

🔮 CINDERELLA CANDIDATES — Lower Seeds with Highest Heat

Seed   Team                       ELO   Record    H-5g    H-3g    H-1g  Margin
────────────────────────────────────────────────────────────────────────────────
Y15    Tennessee St              1581     20-9   +15.3   +14.3   +11.3    +4.1
Y16a   Howard                    1554    19-10   +13.7    +6.9    +7.8    +7.2
Y16b   UMBC                      1589     22-8   +13.7   +16.8   +14.2    +7.5
X16b   Prairie View              1484    14-17   +13.5   +11.7   +20.6    -2.8
W11    South Florida             1635     24-8   +12.0    +8.9   +14.3   +10.7
W13    Cal Baptist               1616     25-8    +9.6   +11.5    +9.0    +5.5
Y13    Hofstra                   1598    22-10    +9.3   +12.1    -0.2    +6.9
W07    UCLA                      1605    23-11    +8.6    +9.8    +7.2    +6.7
W12    Northern Iowa             1569    22-12    +8.3    +9.2    +5.7    +7.0
Y12    Akron                     1660     27-5    +8.0    +6.9    +3.5  

---

## Matchup Explorer

Use the function below to compare any two teams head-to-head. Pass team names (partial match) or TeamIDs.

In [58]:
def compare(team_a, team_b):
    """Compare two teams head-to-head. Pass names (partial match) or integer TeamIDs."""

    def resolve(team):
        if isinstance(team, int):
            return team, tid_to_name.get(team, str(team))
        # Partial name match
        matches = teams.filter(pl.col("TeamName").str.to_lowercase().str.contains(str(team).lower()))
        if matches.height == 0:
            print(f"No team found matching '{team}'")
            return None, None
        if matches.height > 1:
            print(f"Multiple matches for '{team}': {matches['TeamName'].to_list()}")
        tid = int(matches["TeamID"][0])
        return tid, matches["TeamName"][0]

    tid_a, name_a = resolve(team_a)
    tid_b, name_b = resolve(team_b)
    if tid_a is None or tid_b is None:
        return

    low_id, high_id = min(tid_a, tid_b), max(tid_a, tid_b)
    low_name = name_a if tid_a == low_id else name_b
    high_name = name_b if tid_a == low_id else name_a

    # Get features for both teams
    feat_a = m_2026.filter(pl.col("team_id") == tid_a)
    feat_b = m_2026.filter(pl.col("team_id") == tid_b)

    if feat_a.height == 0 or feat_b.height == 0:
        print("One or both teams not found in 2026 M features")
        return

    fa = feat_a.to_dicts()[0]
    fb = feat_b.to_dicts()[0]

    # Get seeds if available
    seed_a = seeds_2026.filter(pl.col("TeamID") == tid_a)
    seed_b = seeds_2026.filter(pl.col("TeamID") == tid_b)
    seed_a_str = seed_a["Seed"][0] if seed_a.height > 0 else "N/A"
    seed_b_str = seed_b["Seed"][0] if seed_b.height > 0 else "N/A"

    p_low = get_pred(tid_a, tid_b)
    p_a = p_low if tid_a == low_id else 1 - p_low

    print(f"{'═' * 70}")
    print(f"  {name_a} ({seed_a_str}) vs {name_b} ({seed_b_str})")
    print(f"{'═' * 70}")
    print(f"\n  P({name_a} wins) = {p_a:.1%}    |    P({name_b} wins) = {1 - p_a:.1%}\n")

    stats = [
        ("Record", f"{fa['wins']}-{fa['losses']}", f"{fb['wins']}-{fb['losses']}"),
        ("Win Rate", f"{fa['win_rate']:.1%}", f"{fb['win_rate']:.1%}"),
        ("Avg Margin", f"{fa['avg_margin']:+.1f}", f"{fb['avg_margin']:+.1f}"),
        ("Avg Pts For", f"{fa['avg_pts_for']:.1f}", f"{fb['avg_pts_for']:.1f}"),
        ("Avg Pts Against", f"{fa['avg_pts_against']:.1f}", f"{fb['avg_pts_against']:.1f}"),
        ("Last 5 Win Rate", f"{fa['last5_win_rate']:.1%}", f"{fb['last5_win_rate']:.1%}"),
        ("Last 5 Margin", f"{fa['last5_avg_margin']:+.1f}", f"{fb['last5_avg_margin']:+.1f}"),
        ("ELO", f"{(fa['season_end_elo'] or 1500):.0f}", f"{(fb['season_end_elo'] or 1500):.0f}"),
        ("Heat (1g)", f"{(fa['pre_tourney_heat_1g'] or 0):+.1f}", f"{(fb['pre_tourney_heat_1g'] or 0):+.1f}"),
        ("Heat (3g)", f"{(fa['pre_tourney_heat_3g'] or 0):+.1f}", f"{(fb['pre_tourney_heat_3g'] or 0):+.1f}"),
        ("Heat (5g)", f"{(fa['pre_tourney_heat_5g'] or 0):+.1f}", f"{(fb['pre_tourney_heat_5g'] or 0):+.1f}"),
    ]

    print(f"  {'Stat':<18} {name_a:>18} {name_b:>18}")
    print(f"  {'─' * 56}")
    for label, val_a, val_b in stats:
        print(f"  {label:<18} {val_a:>18} {val_b:>18}")

    print(f"\n  PICK: {'→ ' + name_a if p_a >= 0.5 else '→ ' + name_b} ({max(p_a, 1 - p_a):.1%})")


# Example: compare two teams
compare("Duke", "Auburn")

══════════════════════════════════════════════════════════════════════
  Duke (W01) vs Auburn (N/A)
══════════════════════════════════════════════════════════════════════

  P(Duke wins) = 90.6%    |    P(Auburn wins) = 9.4%

  Stat                             Duke             Auburn
  ────────────────────────────────────────────────────────
  Record                           32-2              17-16
  Win Rate                        94.1%              51.5%
  Avg Margin                      +19.1               +3.3
  Avg Pts For                      82.3               82.7
  Avg Pts Against                  63.1               79.4
  Last 5 Win Rate                100.0%              40.0%
  Last 5 Margin                   +12.2               +0.8
  ELO                              1738               1515
  Heat (1g)                        +4.7              +15.2
  Heat (3g)                        +2.0               +7.8
  Heat (5g)                       +10.3               -0.4

  PICK

In [59]:
# Try your own matchups! Uncomment or edit these:
# compare("Houston", "Florida")
# compare("Kansas", "Michigan St")
# compare(1385, 1120)  # St John's vs Auburn (by TeamID)
compare("Connecticut", "St John")

══════════════════════════════════════════════════════════════════════
  Connecticut (W02) vs St John's (W05)
══════════════════════════════════════════════════════════════════════

  P(Connecticut wins) = 66.8%    |    P(St John's wins) = 33.2%

  Stat                      Connecticut          St John's
  ────────────────────────────────────────────────────────
  Record                           29-5               28-6
  Win Rate                        85.3%              82.4%
  Avg Margin                      +12.4              +11.6
  Avg Pts For                      77.5               81.6
  Avg Pts Against                  65.1               70.0
  Last 5 Win Rate                 60.0%             100.0%
  Last 5 Margin                    +3.8              +10.6
  ELO                              1669               1685
  Heat (1g)                        +4.9               +3.1
  Heat (3g)                        -0.6               +2.1
  Heat (5g)                        +5.1      

---

## Printable Bracket Summary

Copy-friendly bracket: the model's predicted winners for each round, organized by region.

In [60]:
# Printable bracket — compact summary by region and round
print("╔══════════════════════════════════════════════════════════════════════════╗")
print("║                   2026 NCAA MEN'S TOURNAMENT BRACKET                   ║")
print("║                        Model Predicted Winners                         ║")
print("╚══════════════════════════════════════════════════════════════════════════╝\n")

for round_name in ["Round of 64", "Round of 32", "Sweet 16", "Elite 8", "Final Four", "Championship"]:
    results = round_results.get(round_name, [])
    if not results:
        continue

    print(f"┌{'─' * 74}┐")
    print(f"│  {round_name.upper():<72}│")
    print(f"├{'─' * 74}┤")

    by_region: dict[str, list] = {}
    for r in results:
        rg = r["region"] or "National"
        by_region.setdefault(rg, []).append(r)

    for region, games in sorted(by_region.items()):
        if region != "National":
            print(f"│  {region:<72}│")
        for g in games:
            ws = seed_num(g['winner_seed'])
            ls = seed_num(g['loser_seed'])
            upset_marker = "🔥" if ws > ls else "  "
            line = f"    {upset_marker} ({ws:>2}) {g['winner_name']:<18} over ({ls:>2}) {g['loser_name']:<18} [{g['win_prob']:.0%}]"
            print(f"│{line:<74}│")

    print(f"└{'─' * 74}┘\n")

# Final output
CHAMP_SLOT = "R6CH"
if CHAMP_SLOT in slot_winner:
    champ = slot_winner[CHAMP_SLOT]
    print(f"  🏆 PREDICTED CHAMPION: ({seed_num(champ[2]):>2}) {champ[1]} 🏆\n")

    # Final Four teams
    ff_teams = [r for r in round_results.get("Final Four", [])]
    if ff_teams:
        print("  Final Four:")
        for g in ff_teams:
            print(f"    ({seed_num(g['winner_seed']):>2}) {g['winner_name']} over ({seed_num(g['loser_seed']):>2}) {g['loser_name']}")

print("\n  ℹ️  To adjust predictions, use the ml_walkthrough.ipynb notebook")
print("     to change the Model/Simulation blend weights.")

╔══════════════════════════════════════════════════════════════════════════╗
║                   2026 NCAA MEN'S TOURNAMENT BRACKET                   ║
║                        Model Predicted Winners                         ║
╚══════════════════════════════════════════════════════════════════════════╝

┌──────────────────────────────────────────────────────────────────────────┐
│  ROUND OF 64                                                             │
├──────────────────────────────────────────────────────────────────────────┤
│  East                                                                    │
│       ( 1) Florida            over (16) Prairie View       [98%]         │
│       ( 2) Houston            over (15) Idaho              [97%]         │
│       ( 3) Illinois           over (14) Penn               [92%]         │
│       ( 4) Nebraska           over (13) Troy               [79%]         │
│       ( 5) Vanderbilt         over (12) McNeese St         [64%]         │
│ 